# Config and BPE Tokenizer

In [1]:
from config import ModelConfig

model_cfg = ModelConfig(
    name="BaseFromPaper",
    d_vocab=32_000,
    context_len=24,
    d_k=64,
    d_v=64,
    d_model=64,
    h=8,
    N=6,
    dropout_rate=0.1,
)

In [2]:
%env GLOG_minloglevel=2
from tokenizer import TokenizerBPE

tokenizer = TokenizerBPE(name=f"{model_cfg.d_vocab//1_000}k_shared", vocab_size=model_cfg.d_vocab)

env: GLOG_minloglevel=2


In [3]:
in_en = "some input text"
out_en = tokenizer.encode(in_en)
tokenizer.decode(out_en) == in_en

True

In [4]:
in_pl = "jakiś tekst wejściowy"
out_pl = tokenizer.encode(in_pl)
tokenizer.decode(out_pl) == in_pl

True

# Defining Model

In [5]:
import torch
from model import TransformerSeq2Seq

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

'cuda'

In [6]:
translate_model = TransformerSeq2Seq(model_cfg)

In [7]:
from config import TrainConfig

eval_every_n = 100

train_cfg = TrainConfig(
    epochs=100,
    batch_size=256,
    beta1=0.9,
    beta2=0.98,
    eps=1e-9,
    lr_factor=model_cfg.d_model ** -0.5,
    lr_warmup_steps=4000,
    label_smoothing_eps=0.1,
    eval_every_epochs=eval_every_n,
    eval_max_batches=10,
)

In [8]:
from dataloader import TranslationDataset

start_id=1
end_id=2
pad_id=3

train_dataset = TranslationDataset(
    en_path="./opus100/opus.en-pl-train.en",
    pl_path="./opus100/opus.en-pl-train.pl",
    sp_model=tokenizer._sp,
    start_id=start_id,
    end_id=end_id,
    pad_id=pad_id,
    context_len=model_cfg.context_len,
)

validate_dataset = TranslationDataset(
    en_path="./opus100/opus.en-pl-dev.en",
    pl_path="./opus100/opus.en-pl-dev.pl",
    sp_model=tokenizer._sp,
    start_id=start_id,
    end_id=end_id,
    pad_id=pad_id,
    context_len=model_cfg.context_len,
)

In [9]:
from train import run_training

train_loss, valid_loss = run_training(
    translate_model,
    train_cfg,
    train_dataset,
    validate_dataset,
    pad_id,
    DEVICE,
    max_batch_per_epoch=None,
)

Epoch 1/100: 100%|██████████| 3907/3907 [13:17<00:00,  4.90it/s, loss=5.1254, lr=1.93e-03]


epoch 0: train_loss=6.3123 valid_loss=4.4227


Epoch 2/100:   0%|          | 18/3907 [00:04<14:42,  4.41it/s, loss=5.2268, lr=1.94e-03]


KeyboardInterrupt: 

In [ ]:
# torch.save(translate_model.state_dict, ".")

In [10]:
import matplotlib.pyplot as plt

train_x = [epoch for epoch, loss in train_loss]
train_y = [loss for epoch, loss in train_loss]

valid_x = [epoch for epoch, loss in valid_loss]
valid_y = [loss for epoch, loss in valid_loss]

plt.plot(train_x, train_y, label="train")
plt.plot(valid_x, valid_y, label="valid")

plt.legend()
plt.show()

NameError: name 'train_loss' is not defined